In [227]:
#easier to use notebook when getting started: using this for minor bits as I expand the database
# Krista 28 August 2026

In [1]:
import pandas as pd
import os
import re
import pdb #user with set_trace()

import xml.etree.ElementTree as ET

In [3]:
#parsing out the information in the filenames, use a function
#this is such a mess...but I built this up in pieces and it works for all the files
def parseSample(sample):
    #parsed = {} #use the next row and setup the dictionary first, even empty it is useful as I don't have to repeat all the NA options
    parsed = dict({'cruise5':'',
                   'cast':'',
                   'niskin':'',
                   'cruise':'',
                   'otherInfo':'',
                   'depth':'',
                   'sampleNumber':'',
                   'New_Bottle_ID':'',
                  'seqType':''})
    #start with 10 digits - cruise - cast - niskin and then underscores to note 18S_V4
    ru = [match.start() for match in re.finditer("_",sample)]
    #print(sample)
    if len(ru) ==2:
        parsed['cruise5'] = sample[:5]
        parsed['cast'] = sample[5:8]
        parsed['niskin']= sample[8:10]        
        parsed['New_Bottle_ID'] = sample[:10]
        parsed['otherInfo'] = sample[ru[0]+1:]
        if '18S' in parsed['otherInfo']:
            parsed['seqType'] = 'V4_18s'
        elif '16S_V1V2' in parsed['otherInfo']:
            parsed['seqType'] = 'V1V2'
            
    parsed = pd.DataFrame([parsed]) 

    return parsed

In [4]:
base_dir = 'd:/dropbox/github_niskin/SargassoDB/'
data_dir = 'test_data/'
fName = 'biosample_result.xml'

tree = ET.parse(os.path.join(base_dir,data_dir,fName))
root = tree.getroot()

parsed_records = []

for biosample in root.findall(".//BioSample"):
    sample_id = biosample.attrib.get("id")
    
    # Initialize variables for this sample
    sample_name = None
    biosample_accession = None
    sra_id = None
    
    # Loop over the IDs section
    for id_tag in biosample.findall(".//Ids/Id"):
        db_type = id_tag.attrib.get("db")
        db_label = id_tag.attrib.get("db_label")
        is_primary = id_tag.attrib.get("is_primary")
        
        # 1. Capture the Sample Name
        if db_label == "Sample name":
            sample_name = id_tag.text
            
        # 2. Capture the Primary Accession (SAMN...)
        elif db_type == "BioSample" and is_primary == "1":
            biosample_accession = id_tag.text
            
        # 3. Capture the SRA ID (SRS...)
        elif db_type == "SRA":
            sra = id_tag.text
            
    # 3. Pull target keys out of the <Attributes> block
    for attr in biosample.findall(".//Attributes/Attribute"):
        attr_name = attr.attrib.get("attribute_name")
        
        if attr_name == "collection_date":
            collection_date = attr.text
        elif attr_name == "depth":
            depth = attr.text
        elif attr_name == "temperature":
            temp = attr.text
        elif attr_name == "salinity":
            sal = attr.text
                      
    # 2. Extract Contact Name from Owner information
    contact_node = biosample.find(".//Owner/Contacts/Contact/Name")
    if contact_node is not None:
        first_node = contact_node.find("First")
        last_node = contact_node.find("Last")
        
        # Pull text if nodes exist, then join them cleanly
        first_name = first_node.text if first_node is not None else ""
        last_name = last_node.text if last_node is not None else ""
        contact_name = f"{first_name} {last_name}".strip()
        
            
    # Append collected data to our list
    parsed_records.append({
        "id": sample_id,
        "biosample": biosample_accession,
        "sample": sample_name,
        "dateCollected":collection_date,
        "depth": depth,
        "temp":temp,
        "sal":sal,
        "sra": sra,
        "submitter":contact_name
    })

# Convert to DataFrame and view/save
dfNCBI = pd.DataFrame(parsed_records)
#df.to_csv("../test_data/biosamples_with_sra.csv", index=False)
print(dfNCBI)

            id     biosample               sample dateCollected    depth  \
0     44822555  SAMN44822555         BSV4_745_S92    2021-03-24  299.757   
1     44822554  SAMN44822554         BSV4_744_S91    2021-03-24  249.722   
2     44822553  SAMN44822553         BSV4_743_S90    2021-03-24  199.364   
3     44822552  SAMN44822552         BSV4_742_S89    2021-03-24  159.281   
4     44822551  SAMN44822551         BSV4_741_S88    2021-03-24   120.23   
...        ...           ...                  ...           ...      ...   
1406  22169535  SAMN22169535  9161400209_16S_V1V2    2016-07-09      160   
1407  22169534  SAMN22169534  9161400207_16S_V1V2    2016-07-09      120   
1408  22169533  SAMN22169533  9161400205_16S_V1V2    2016-07-09       80   
1409  22169532  SAMN22169532  9161400203_16S_V1V2    2016-07-09       40   
1410  22169531  SAMN22169531  9161400201_16S_V1V2    2016-07-09        1   

         temp      sal          sra        submitter  
0     16.9276  36.2961  SRS23255

In [231]:
#first up, see if we have any duplicates...some people used nominal depth and some used actual depths.
#Also, have V4_18s and what I think is V1V2. There is also V4_16s data in here (samples BSV4)
#also note Leo has four samples with depth=-999 (cruise 10329)
#df[df['depth'] < 0]

In [5]:
#Fabian already has a list of NCBI details, use that to see if we have found everything (and pull in sample details)
fNameLOGncbi = 'BIOS-SCOPE-NCBI_Log_Nov2024.xlsx'
dfLOGncbi = pd.DataFrame(pd.read_excel(os.path.join(base_dir,data_dir,fNameLOGncbi)))
#tidy up - make sure New_Bottle_ID is an integer
dfLOGncbi['New_Bottle_ID'] = dfLOGncbi['New_Bottle_ID'].astype('Int64')

In [7]:
dfLOGncbi = dfLOGncbi['Biosample'].value_counts().reset_index()
dfLOGncbi[dfLOGncbi['count']>1]
#The duplicate here for the Biosample is legit - Fabian and Nicole have two sequences from the same sample

,Biosample,count
0,check discrepancy,8
1,SAMN22169720,2


In [236]:
#now, while I am at it, pull in the data from the LTT paper (probably should ultimately be a different script, but good for testing)
fNameLTT = 'LTTpaper/Table_S1_Accession_SampleID_numbers.xlsx'
dfLTTtableS1 = pd.DataFrame(pd.read_excel(os.path.join(base_dir,data_dir,fNameLTT),skiprows=1))
#tidy up - make sure New_Bottle_ID is an integer
dfLTTtableS1['Sample.ID'] = dfLTTtableS1['Sample.ID'].astype('Int64')
dfLTTtableS1[['Year','Month','Depth']] = dfLTTtableS1[['Year','Month','Depth']].astype('Int64')


In [237]:
#there is a second table for the LTT paper with the deep sequencing:
fNameLTTdeep = 'LTTpaper/Table_S11_Accession_Deep_Seq.xlsm'
dfLTTdeep = pd.DataFrame(pd.read_excel(os.path.join(base_dir,data_dir,fNameLTTdeep)))
#tidy up - make sure New_Bottle_ID is an integer
dfLTTdeep['Sample.ID'] = dfLTTdeep['Sample.ID'].astype('Int64')
dfLTTdeep[['Year','Month','Depth']] = dfLTTdeep[['Year','Month','Depth']].astype('Int64')

In [102]:
# df = pd.merge(fNameLOGncbi,dfNCBI,how = 'left',left_on='biosample',right_on='Biosample')

In [ ]:
# for idx,row in dfNCBI.iterrows():
#     #only do something if New_Bottle_ID is empty
#     if pd.isna(row['New_Bottle_ID']) & (row['submitter'] != 'Ben Temperton'):        
#         one = row['sample']
#         parsed = parseSample(one)
#         #pdb.set_trace()
#         dfNCBI.at[idx,'New_Bottle_ID'] = parsed['New_Bottle_ID']

In [146]:
#tidy up - make sure New_Bottle_ID is an integer
df['New_Bottle_ID'] = df['New_Bottle_ID'].astype('Int64')

In [147]:
df['zSearch'] = pd.to_numeric(df['depth'],errors='coerce').round(0).astype('Int64')
dp = df.pop('zSearch')
ii = df.columns.get_loc('depth') + 1
df.insert(ii,'zRound',dp)

ValueError: cannot insert zRound, already exists

In [66]:
#there is only one case where a biosample is used twice, this is a duplicate sample in Nicole and Fabian's dataset
dfc = df['biosample'].value_counts().reset_index()
dfc[dfc['count']>1]

# dfc = df['New_Bottle_ID'].value_counts().reset_index()
# dfc[dfc['count']>1]

,biosample,count
0,SAMN22169720,2


In [105]:
#and now it is clear that there are multiple biosamples for some New_ID (18S v. V1V2 and V4_16s)
#Need to pull the NewID from samples at NCBI without that information

In [160]:
dfc = df['New_Bottle_ID'].value_counts().reset_index()
dfc[dfc['count']>1]

,New_Bottle_ID,count
0,1033201509,3
1,1033401205,2
2,1033201501,2
3,1035202106,2
4,1035202108,2
...,...,...
279,1035202113,2
280,1035202111,2
281,1035402011,2
282,1035402013,2


In [163]:
#there are way too many samples with temperature at the same value (e.g., 21.142, which happens 1180 !)
len(df[df['temp'] == '21.142'])

#This is clearly a problem...oddly salinity varies a little more

1180

In [176]:
dft.head()

,Sample.ID,Year,Month,Depth,BioSample,SRA_16S_V1V2,Reads_raw,Reads_trim_filt,reads_merge,reads_nonchim_final
0,1003500112,1991,8,0,SAMN52634534,SRR35786328,93227,86441,78117,76281
1,1003600312,1991,9,0,SAMN52634535,SRR35786329,86281,79345,71337,69188
2,1003700112,1991,10,0,SAMN52634536,SRR35786133,87087,79741,71629,69138
3,1003800112,1991,11,0,SAMN52634537,SRR35786025,64460,58507,51384,49680
4,1003900112,1991,12,0,SAMN52634538,SRR35786055,70614,64449,56578,54214


In [ ]:
#existing dates are yyyy-mm-dd ...but the LTT paper does not list the day (but has ID, so pull that info from NewID)
#dft['Year'].astype(str) + '-' + dft['Month'].astype(str) + '-' + dft['Day'].astype(str)
#later

In [185]:
# Returns rows in df['A'] that are NOT found anywhere in df['B']
# df['A'][~df['A'].isin(df['B'])]

#df['New_Bottle_ID'][~df['New_Bottle_ID'].isin(dft['Sample.ID'])]

dft['Sample.ID'][~dft['Sample.ID'].isin(df['New_Bottle_ID'])]

# dft['Sample.ID'])
# setB = set(df['New_Bottle_ID'])

0      1003500112
1      1003600312
2      1003700112
3      1003800112
4      1003900112
          ...    
384    1029100811
385    2029100212
386    1029200816
437    2016200201
438    2016200221
Name: Sample.ID, Length: 344, dtype: Int64

In [186]:
df['New_Bottle_ID'][~df['New_Bottle_ID'].isin(dft['Sample.ID'])]

0       2037900119
1       2037900117
2       2037900115
3       2037900113
4       2037900110
           ...    
1407    9161400209
1408    9161400207
1409    9161400205
1410    9161400203
1411    9161400201
Name: New_Bottle_ID, Length: 1232, dtype: Int64

In [188]:
df['New_Bottle_ID'].isin(dft['Sample.ID'])

0       False
1       False
2       False
3       False
4       False
        ...  
1407    False
1408    False
1409    False
1410    False
1411    False
Name: New_Bottle_ID, Length: 1412, dtype: boolean

In [199]:
df['match']  = dft['Sample.ID'].isin(df['New_Bottle_ID'])

In [201]:
df[df['match']==True]

,id,biosample,sample,dateCollected,depth,zRound,temp,sal,sra,submitter,...,Reference (1st used in),Sample name V4,SRA_16S_V4,Status_V4_16S,Reference (1st used in).1,V4_Sequencing_File,SRA_18S_V4,Status,Reference,match
172,44822383,SAMN44822383,20345_300_S113,2018-03-25,301.897,302,17.6386309,36.43233987,SRS23255864,Fabian Wittmers,...,NaN,20345_300_S113,SRR31399217,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s113_index_GTAACGAG_ATCGTACG_20345_300_S113,NaN,NaN,NaN,True
173,44822382,SAMN44822382,20345_250_S112,2018-03-25,248.788,249,18.0409309,36.48696127,SRS23255862,Fabian Wittmers,...,NaN,20345_250_S112,SRR31399219,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s112_index_GATCTACG_GACACCGT_20345_250_S112,NaN,NaN,NaN,True
174,44822381,SAMN44822381,20344_300_S108,2018-02-26,301.174,301,17.91049877,36.47699056,SRS23255863,Fabian Wittmers,...,NaN,20344_300_S108,SRR31399220,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s108_index_GATCTACG_CTGCGTGT_20344_300_S108,NaN,NaN,NaN,True
175,44822380,SAMN44822380,20344_250_S107,2018-02-26,253.354,253,18.36029877,36.52839979,SRS23255859,Fabian Wittmers,...,NaN,20344_250_S107,SRR31399221,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s107_index_GATCTACG_TAGCGAGT_20344_250_S107,NaN,NaN,NaN,True
176,44822379,SAMN44822379,10344_300_S22,2018-02-12,301.232,301,18.37098211,36.53665817,SRS23255861,Fabian Wittmers,...,NaN,10344_300_S22,SRR31399222,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s022_index_GACATAGT_CTACTATA_10344_300_S22,NaN,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
435,36323009,SAMN36323009,10362_40,2019-08-21,40,40,21.142,36.40605727,SRS18186397,Leocadio Blanco-Bercial,...,"Eckmann et al., 2024",BSv4_535_S14,SRR31627470,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s014_indexN715_C_S515_C_ATCTCAGG_TTCTAGC...,SRR25161141,"uploaded, private",https://doi.org/10.1101/2023.06.29.547096,True
436,36323008,SAMN36323008,10362_300,2019-08-21,300,300,21.142,36.64291056,SRS18186396,Leocadio Blanco-Bercial,...,NaN,BSv4_541_S20,SRR31627464,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s020_indexN701_C_S522_C_TAAGGCGA_TTATGCG...,SRR25161232,"uploaded, private",https://doi.org/10.1101/2023.06.29.547102,True
439,36323005,SAMN36323005,10362_160,2019-08-21,160,160,21.142,36.71068659,SRS18186488,Leocadio Blanco-Bercial,...,"Eckmann et al., 2024",BSv4_538_S17,SRR31627467,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s017_indexN701_C_S518_C_TAAGGCGA_CTATTAA...,SRR25161139,"uploaded, private",https://doi.org/10.1101/2023.06.29.547099,True
440,36323004,SAMN36323004,10362_120,2019-08-21,120,120,21.142,36.75571696,SRS18186487,Leocadio Blanco-Bercial,...,"Eckmann et al., 2024",BSv4_537_S16,SRR31627468,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s016_indexN715_C_S517_C_ATCTCAGG_GCGTAAG...,SRR25161226,"uploaded, private",https://doi.org/10.1101/2023.06.29.547098,True


In [ ]:
## But looking at this, something is wrong with my logic

In [203]:
df[df['New_Bottle_ID'] == 1032501412]
#this will list two from the NCBI list in Google Drive, one of which overlaps with the LTT paper list...but the LTT paper lists another biosample, but I cannot find that anywhere:
#SAMN52634884

,id,biosample,sample,dateCollected,depth,zRound,temp,sal,sra,submitter,...,Reference (1st used in),Sample name V4,SRA_16S_V4,Status_V4_16S,Reference (1st used in).1,V4_Sequencing_File,SRA_18S_V4,Status,Reference,match
208,44822347,SAMN44822347,10325_200_S35,2016-06-16,201.8,202,18.652,36.484,SRS23255799,Fabian Wittmers,...,NaN,10325_200_S35,SRR31399281,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s035_index_TGAGTACG_TAGCGAGT_10325_200_S35,NaN,NaN,NaN,True
1010,22786352,SAMN22786352,1032501412_18S_V4,2016-06-16,201.8,202,21.142,36.484,SRS10846832,Leocadio Blanco-Bercial,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [206]:
df.loc[1010,]

id                                          22786352
biosample                               SAMN22786352
sample                             1032501412_18S_V4
dateCollected                             2016-06-16
depth                                          201.8
zRound                                           202
temp                                          21.142
sal                                           36.484
sra                                      SRS10846832
submitter                    Leocadio Blanco-Bercial
Project                                          NaN
Cruise_ID                                        NaN
Cruise                                           NaN
Site                                             NaN
Depths                                           NaN
Sample name V1V2                                 NaN
Date_M_D_Y                                       NaN
Cast                                             NaN
Niskin                                        

In [204]:
df[df['biosample'] == 'SAMN52634884']

,id,biosample,sample,dateCollected,depth,zRound,temp,sal,sra,submitter,...,Reference (1st used in),Sample name V4,SRA_16S_V4,Status_V4_16S,Reference (1st used in).1,V4_Sequencing_File,SRA_18S_V4,Status,Reference,match


In [205]:
df[df['biosample'] == 'SAMN44822347']

,id,biosample,sample,dateCollected,depth,zRound,temp,sal,sra,submitter,...,Reference (1st used in),Sample name V4,SRA_16S_V4,Status_V4_16S,Reference (1st used in).1,V4_Sequencing_File,SRA_18S_V4,Status,Reference,match
208,44822347,SAMN44822347,10325_200_S35,2016-06-16,201.8,202,18.652,36.484,SRS23255799,Fabian Wittmers,...,NaN,10325_200_S35,SRR31399281,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s035_index_TGAGTACG_TAGCGAGT_10325_200_S35,NaN,NaN,NaN,True


In [211]:
dft[dft['BioSample'] == dfd.loc[0,'BioSample']]

,Sample.ID,Year,Month,Depth,BioSample,SRA_16S_V1V2,Reads_raw,Reads_trim_filt,reads_merge,reads_nonchim_final
0,1003500112,1991,8,0,SAMN52634534,SRR35786328,93227,86441,78117,76281


In [212]:
both = pd.merge(dfd,dft,left_on='BioSample',right_on='BioSample',how = 'left')

In [213]:
both

,title,Sample.ID_x,Year_x,Month_x,Depth_x,BioSample,SRA_16S_V1V2_x,Reads In,Filter Trim Reads,Merged Reads,...,Unnamed: 11,Sample.ID_y,Year_y,Month_y,Depth_y,SRA_16S_V1V2_y,Reads_raw,Reads_trim_filt,reads_merge,reads_nonchim_final
0,35_0M_S1,1003500112,1991,8,0,SAMN52634534,SRR36513417,1.219479e+06,1.075919e+06,9.897750e+05,...,NaN,1003500112,1991,8,0,SRR35786328,93227.0,86441.0,78117.0,76281.0
1,42_0M_S3,1004200312,1992,3,0,SAMN52634541,SRR36513416,1.277756e+06,1.139673e+06,1.065760e+06,...,NaN,1004200312,1992,3,0,SRR35786206,87587.0,80340.0,70882.0,67198.0
2,46_0M_S5,1004600212,1992,7,0,SAMN52634545,SRR36513405,1.527606e+06,1.339331e+06,1.248835e+06,...,NaN,1004600212,1992,7,0,SRR35786341,77837.0,71595.0,64563.0,62715.0
3,47_0M_S17,1004700312,1992,8,0,SAMN52634546,SRR36513402,7.318940e+05,6.421660e+05,5.904730e+05,...,NaN,1004700312,1992,8,0,SRR35786353,60281.0,55975.0,50113.0,48693.0
4,56_0M_S15,1005600312,1993,5,0,SAMN52634555,SRR36513401,6.232920e+05,5.581620e+05,5.053150e+05,...,NaN,1005600312,1993,5,0,SRR35786287,70594.0,65742.0,57581.0,53025.0
5,65_0M_S19,1006500601,1994,2,0,SAMN52634562,SRR36513400,6.267540e+05,5.440870e+05,4.982340e+05,...,NaN,1006500601,1994,2,0,SRR35786045,70596.0,63980.0,55012.0,52571.0
6,332_0M_S7,1033201501,2017,1,0,SAMN22169716,SRR36513399,1.281895e+06,1.151551e+06,1.079384e+06,...,NaN,1033201501,2017,1,1,SRR16297311,97825.0,88337.0,75951.0,70088.0
7,350_0M_S11,1035001601,2018,8,0,SAMN28811517,SRR36513398,1.269220e+06,1.158599e+06,1.094974e+06,...,NaN,1035001601,2018,8,1,SRR19520206,83305.0,72348.0,64572.0,61156.0
8,362_0M_S23,1036202201,2019,8,0,SAMN36323002,SRR36513397,9.270070e+05,8.136150e+05,7.571140e+05,...,NaN,1036202201,2019,8,1,SRR27540812,85439.0,72149.0,63066.0,60390.0
9,371_0M_S27,1037101511,2020,8,0,SAMN52634710,SRR36513396,6.012340e+05,5.431120e+05,5.052020e+05,...,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN


In [226]:
dft[dft['Sample.ID'] == 1003500112]

,Sample.ID,Year,Month,Depth,BioSample,SRA_16S_V1V2,Reads_raw,Reads_trim_filt,reads_merge,reads_nonchim_final
0,1003500112,1991,8,0,SAMN52634534,SRR35786328,93227,86441,78117,76281


In [222]:
dft[dft['BioSample'] == 'SAMN52634905']

,Sample.ID,Year,Month,Depth,BioSample,SRA_16S_V1V2,Reads_raw,Reads_trim_filt,reads_merge,reads_nonchim_final


In [223]:
df[df['New_Bottle_ID'] == 1037101511]

,id,biosample,sample,dateCollected,depth,zRound,temp,sal,sra,submitter,...,Reference (1st used in),Sample name V4,SRA_16S_V4,Status_V4_16S,Reference (1st used in).1,V4_Sequencing_File,SRA_18S_V4,Status,Reference,match
89,44822466,SAMN44822466,BSv4_647_S126,2020-08-19,199.829,200,19.736,36.6859,SRS23255821,Fabian Wittmers,...,NaN,BSv4_647_S126,SRR31399261,uploaded; private,Wittmers & Dames et al. (in prep.),lane1_s126_indexN714_A_S502_A_GCTCATGA_CTCTCTA...,NaN,NaN,NaN,False


In [225]:
df[df['New_Bottle_ID'] == 1003500112]

,id,biosample,sample,dateCollected,depth,zRound,temp,sal,sra,submitter,...,Reference (1st used in),Sample name V4,SRA_16S_V4,Status_V4_16S,Reference (1st used in).1,V4_Sequencing_File,SRA_18S_V4,Status,Reference,match


In [202]:
df.to_excel('../test_data/out5.xlsx')

In [ ]:
#Stick some code below this spot as a holding zone
raise SystemExit("Stop execution here")

In [ ]:
df['zSearch'] = pd.to_numeric(df['depth'],errors='coerce')
ud = df['zSearch'].round(0).astype('Int64')
ud.unique()

In [3]:
os.getcwd()

'd:\\dropbox\\github_niskin\\SargassoDB\\notebooks'